# Notebook 18 - Video Grounding, Summarization, and Event Extraction

This notebook moves from sampled frames to event-level outputs: clip summaries, grounded spans, and simple event extraction.


## What you will learn

- how to represent events as frame or time spans
- how to build timeline summaries that preserve evidence
- how to score event extraction quality


In [ ]:
!pip install -q numpy pandas matplotlib
print("Installed notebook dependencies.")


In [ ]:
import json
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-multimodal")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "18_video_grounding_summarization_and_event_extraction"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(7)
np.random.seed(7)



print("Cache dir:", CACHE_DIR)
print("Artifact root:", NOTEBOOK_ROOT.resolve())


## Step 1: Propose event windows

Event windows are the video analogue of bounding boxes and timestamp spans.


In [ ]:
event_windows = pd.DataFrame([
    {"event_id": "e1", "start_s": 2.0, "end_s": 4.0, "description": "user opens reset form"},
    {"event_id": "e2", "start_s": 5.0, "end_s": 8.0, "description": "error banner appears after submit"},
])
event_windows


## Step 2: Build a grounded summary

A useful video summary cites the event window that supports each sentence.


In [ ]:
grounded_summary = [
    {"sentence": "The user opens the reset form.", "evidence": "2.0-4.0s"},
    {"sentence": "After submit, an error banner appears and blocks progress.", "evidence": "5.0-8.0s"},
]
print(json.dumps(grounded_summary, indent=2))


## Step 3: Score event extraction

IoU-style overlap is a simple way to compare predicted event spans to reference spans.


In [ ]:
def span_iou(a_start, a_end, b_start, b_end):
    intersection = max(0, min(a_end, b_end) - max(a_start, b_start))
    union = max(a_end, b_end) - min(a_start, b_start)
    return intersection / union if union else 0.0

predicted = {"start_s": 5.5, "end_s": 7.5}
reference = {"start_s": 5.0, "end_s": 8.0}
print("Event IoU:", round(span_iou(predicted["start_s"], predicted["end_s"], reference["start_s"], reference["end_s"]), 3))


## Exercises and extensions

1. Add multiple predicted windows and compute precision at an IoU threshold.
1. Create a summary grader that checks whether each sentence cites a valid event span.
